# 111 — Delta-ML with Enhanced Features

Same 3-tier similarity structure as nb104 but with richer delta features:
- ECFP6 (radius-3 Morgan) difference bits
- MACCS keys difference bits
- Extended physchem (11 props vs 7)
- Scaffold bit (same/different scaffold)
- Ring count difference

Goal: beat nb104 OOF RAE = 0.2772.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, rdMolDescriptors
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM_BASE = dict(n_estimators=1200, num_leaves=64, learning_rate=0.04,
                 min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                 reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
print("imports OK")

imports OK


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} r={pr:.4f} rho={sp:.4f}{ca}")
    return m

In [3]:
# --- Enhanced fingerprint batch functions ---
def ecfp6_batch(smiles_list, n_bits=2048):
    """ECFP6 (radius-3) Morgan fingerprints."""
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=n_bits)
            fps.append(list(fp))
        else:
            fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

def maccs_batch(smiles_list):
    """MACCS 166-bit keys."""
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = MACCSkeys.GenMACCSKeys(mol)
            fps.append(list(fp))
        else:
            fps.append([0]*167)
    return np.array(fps, dtype=np.float32)[:, 1:]  # drop bit 0 (unused)

PHYS_PROPS = ["mw", "logp", "tpsa", "hbd", "hba", "rotbonds", "fsp3",
              "n_rings", "n_aromatic_rings", "heavy_atoms", "formal_charge"]

def physchem_batch(smiles_list):
    rows = []
    for s in smiles_list:
        p = compute_physchem(str(s))
        rows.append([p.get(k, 0) or 0 for k in PHYS_PROPS])
    return np.array(rows, dtype=np.float32)

In [4]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
scaffold_arr = np.array(scaffolds)
te_scaffolds = te["smiles"].map(bemis_murcko).tolist()
te_scaffold_arr = np.array(te_scaffolds)

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))

print("Computing Morgan ECFP4...", flush=True)
fps4_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps4_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)

print("Computing ECFP6...", flush=True)
fps6_tr = ecfp6_batch(tr["smiles"].tolist())
fps6_te = ecfp6_batch(te["smiles"].tolist())

print("Computing MACCS...", flush=True)
maccs_tr = maccs_batch(tr["smiles"].tolist())
maccs_te = maccs_batch(te["smiles"].tolist())

print("Computing physchem...", flush=True)
phys_tr = physchem_batch(tr["smiles"].tolist())
phys_te = physchem_batch(te["smiles"].tolist())

cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Scaffolds {len(set(scaffolds))}  Cliffs {len(cliff_pairs)}")

Computing Morgan ECFP4...


Computing ECFP6...


[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerat

[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerat

[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerator
[03:26:29] DEPRECATION WARNING: please use MorganGenerat

...

[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerat

Computing MACCS...


[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerator
[03:26:36] DEPRECATION WARNING: please use MorganGenerat

Computing physchem...


Train 4,139  Test 513  Scaffolds 3676  Cliffs 0


In [5]:
# --- Tanimoto similarity (ECFP4 for consistency with nb104) ---
print("Computing pairwise Tanimoto...", flush=True)
dot_tt = (fps4_tr @ fps4_tr.T).astype(np.float32)
rowsum = fps4_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

# Test-train similarity
dot_te = (fps4_te @ fps4_tr.T).astype(np.float32)
rs_te = fps4_te.sum(1)[:,None]; rs_tr_v = fps4_tr.sum(1)[None,:]
sim_te_tr = dot_te / np.maximum(rs_te + rs_tr_v - dot_te, 1e-6)

print(f"Train-train sim matrix: {tanimoto_tr.shape}")
print(f"Test-train sim matrix:  {sim_te_tr.shape}")

Computing pairwise Tanimoto...


Train-train sim matrix: (4139, 4139)
Test-train sim matrix:  (513, 4139)


In [6]:
# --- Enhanced delta feature construction ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def compress_maccs(fp, out_dim=32):
    """MACCS is 166 bits, compress to 32."""
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_enhanced_delta_feats(fp4_anchor, fp4_query, fp6_anchor, fp6_query,
                              maccs_anchor, maccs_query,
                              sim_col, anchor_pec50, phys_diff):
    # ECFP4 common + diff (64+64)
    fp4_common = np.minimum(fp4_anchor, fp4_query).astype(np.float32)
    fp4_diff   = np.abs(fp4_anchor - fp4_query).astype(np.float32)
    c4 = compress_fp(fp4_common)
    d4 = compress_fp(fp4_diff)
    # ECFP6 common + diff (64+64)
    fp6_common = np.minimum(fp6_anchor, fp6_query).astype(np.float32)
    fp6_diff   = np.abs(fp6_anchor - fp6_query).astype(np.float32)
    c6 = compress_fp(fp6_common)
    d6 = compress_fp(fp6_diff)
    # MACCS diff (32)
    maccs_diff = np.abs(maccs_anchor - maccs_query).astype(np.float32)
    cm = compress_maccs(maccs_diff)
    # Stack: 64+64+64+64+32+1+1+11 = 301 features
    return np.hstack([c4, d4, c6, d6, cm, sim_col, anchor_pec50[:,None], phys_diff])

print(f"Enhanced delta feature dim: checking...")
_test = make_enhanced_delta_feats(
    fps4_tr[:2], fps4_tr[2:4], fps6_tr[:2], fps6_tr[2:4],
    maccs_tr[:2], maccs_tr[2:4],
    np.ones((2,1)), y_tr[:2], phys_tr[:2]-phys_tr[2:4]
)
print(f"Feature shape: {_test.shape[1]} dims")

Enhanced delta feature dim: checking...
Feature shape: 301 dims


In [7]:
# --- 3-Tier training pairs (same boundaries as nb104) ---
# HIGH:  [0.60, 0.90]  — separate LGBM model
# MED:   [0.45, 0.60)  — separate LGBM model
# LOW:   [0.35, 0.45)  — separate LGBM model
TIERS = {
    "HIGH": (0.60, 0.90),
    "MED":  (0.45, 0.60),
    "LOW":  (0.35, 0.45),
}

def build_tier_data(tier_name, sim_lo, sim_hi, i_idx_all, j_idx_all, sim_all):
    mask = (sim_all >= sim_lo) & (sim_all < sim_hi)
    if tier_name == "HIGH":  # inclusive upper
        mask = (sim_all >= sim_lo) & (sim_all <= sim_hi)
    ii, jj = i_idx_all[mask], j_idx_all[mask]
    sim_ij = sim_all[mask][:,None]
    phys_diff_ij = phys_tr[jj] - phys_tr[ii]
    F_ij = make_enhanced_delta_feats(
        fps4_tr[ii], fps4_tr[jj], fps6_tr[ii], fps6_tr[jj],
        maccs_tr[ii], maccs_tr[jj],
        sim_ij, y_tr[ii], phys_diff_ij)
    F_ji = make_enhanced_delta_feats(
        fps4_tr[jj], fps4_tr[ii], fps6_tr[jj], fps6_tr[ii],
        maccs_tr[jj], maccs_tr[ii],
        sim_ij, y_tr[jj], -phys_diff_ij)
    F  = np.vstack([F_ij, F_ji])
    y  = np.concatenate([y_tr[jj]-y_tr[ii], y_tr[ii]-y_tr[jj]])
    return F, y, len(ii)

# Global pair indices
i_idx_global, j_idx_global = np.where(np.triu(tanimoto_tr > 0.30, k=1))
sim_global = tanimoto_tr[i_idx_global, j_idx_global]
print(f"All train pairs (sim>0.30): {len(i_idx_global):,}")

for tier, (lo, hi) in TIERS.items():
    _, _, n = build_tier_data(tier, lo, hi, i_idx_global, j_idx_global, sim_global)
    print(f"  {tier} [{lo},{hi}]: {n:,} pairs")

All train pairs (sim>0.30): 15,594
  HIGH [0.6,0.9]: 117 pairs


  MED [0.45,0.6]: 910 pairs


  LOW [0.35,0.45]: 4,150 pairs


In [8]:
# --- Train global delta models per tier ---
DELTA_LGBM = dict(n_estimators=800, num_leaves=63, learning_rate=0.05,
                  min_child_samples=15, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

tier_models = {}
for tier, (lo, hi) in TIERS.items():
    F, y, n_pairs = build_tier_data(tier, lo, hi, i_idx_global, j_idx_global, sim_global)
    print(f"Training {tier} delta model on {len(F):,} pairs...", flush=True)
    m = lgb.LGBMRegressor(**DELTA_LGBM)
    m.fit(F, y, callbacks=[lgb.log_evaluation(-1)])
    tier_models[tier] = m
    print(f"  {tier} done.", flush=True)

Training HIGH delta model on 234 pairs...


  HIGH done.


Training MED delta model on 1,820 pairs...


  MED done.


Training LOW delta model on 8,300 pairs...


  LOW done.


In [9]:
# --- Prediction function (per-molecule, best available tier) ---
K_NEIGHBORS = 10

def predict_enhanced(fps4_q, fps4_ref, fps6_q, fps6_ref, maccs_q, maccs_ref,
                     y_ref, phys_q, phys_ref, sim_matrix, fallback_preds):
    N = len(fps4_q)
    preds = np.full(N, np.nan)
    tier_counts = {t: 0 for t in TIERS}
    tier_counts["fallback"] = 0
    tiers_used = np.full(N, "fallback", dtype=object)

    for qi in range(N):
        sim_row = sim_matrix[qi]
        assigned = False

        for tier, (lo, hi) in TIERS.items():
            if tier == "HIGH":
                cand_mask = (sim_row >= lo) & (sim_row <= hi)
            else:
                cand_mask = (sim_row >= lo) & (sim_row < hi)
            cand_idx = np.where(cand_mask)[0]
            if len(cand_idx) == 0:
                continue

            # Top-K by similarity
            top_k = np.argsort(-sim_row[cand_idx])[:K_NEIGHBORS]
            sel_idx = cand_idx[top_k]
            cand_sims = sim_row[sel_idx]

            fp4_q_rep = np.tile(fps4_q[qi:qi+1], (len(sel_idx), 1))
            fp6_q_rep = np.tile(fps6_q[qi:qi+1], (len(sel_idx), 1))
            maccs_q_rep = np.tile(maccs_q[qi:qi+1], (len(sel_idx), 1))
            phys_d = phys_q[qi:qi+1] - phys_ref[sel_idx]
            F_k = make_enhanced_delta_feats(
                fps4_ref[sel_idx], fp4_q_rep,
                fps6_ref[sel_idx], fp6_q_rep,
                maccs_ref[sel_idx], maccs_q_rep,
                cand_sims[:,None], y_ref[sel_idx], phys_d)
            delta_k = tier_models[tier].predict(F_k)
            template_preds = y_ref[sel_idx] + delta_k
            weights = cand_sims ** 2
            preds[qi] = np.average(template_preds, weights=weights)
            tier_counts[tier] += 1
            tiers_used[qi] = tier
            assigned = True
            break  # use best available tier

        if not assigned:
            preds[qi] = fallback_preds[qi]
            tier_counts["fallback"] += 1

    return preds, tier_counts, tiers_used

print("Prediction function ready.")

Prediction function ready.


In [10]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_delta = np.full(len(y_tr), np.nan)
oof_direct = np.full(len(y_tr), np.nan)
oof_tier = np.full(len(y_tr), "fallback", dtype=object)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct LGBM
    m_dir = lgb.train(LGBM_BASE, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(60,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    # Compute sim matrix for this fold
    fps4_va = fps4_tr[va_idx]; fps4_ft = fps4_tr[tr_idx]
    dot_vf = (fps4_va @ fps4_ft.T).astype(np.float32)
    rs_v = fps4_va.sum(1)[:,None]; rs_f = fps4_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    preds_d, tc, tu = predict_enhanced(
        fps4_va, fps4_ft,
        fps6_tr[va_idx], fps6_tr[tr_idx],
        maccs_tr[va_idx], maccs_tr[tr_idx],
        y_tr[tr_idx], phys_tr[va_idx], phys_tr[tr_idx],
        sim_vf, oof_direct[va_idx])
    oof_delta[va_idx] = preds_d
    oof_tier[va_idx] = tu

    r_dir  = rae(y_tr[va_idx], oof_direct[va_idx])
    r_dlt  = rae(y_tr[va_idx], oof_delta[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  delta={r_dlt:.4f}  tiers={tc}", flush=True)

m_dir = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_dlt = full_metrics(y_tr, oof_delta, cliff_pairs, "enhanced_delta_3tier")
print(f"\nTier usage: {dict(zip(list(TIERS.keys())+['fallback'], [int((oof_tier==t).sum()) for t in list(TIERS.keys())+['fallback']]))}")


=== Scaffold 5-fold CV ===


  fold 1  direct=0.4920  delta=0.2195  tiers={'HIGH': 12, 'MED': 177, 'LOW': 344, 'fallback': 295}


  fold 2  direct=0.5734  delta=0.2399  tiers={'HIGH': 15, 'MED': 170, 'LOW': 332, 'fallback': 311}


  fold 3  direct=0.5979  delta=0.2756  tiers={'HIGH': 18, 'MED': 151, 'LOW': 323, 'fallback': 336}


  fold 4  direct=0.5647  delta=0.2567  tiers={'HIGH': 16, 'MED': 164, 'LOW': 329, 'fallback': 319}


  fold 5  direct=0.5954  delta=0.2591  tiers={'HIGH': 15, 'MED': 161, 'LOW': 328, 'fallback': 323}


  [direct_lgbm] RAE=0.5598 MAE=0.5093 R2=0.6075 r=0.7794 rho=0.7337
  [enhanced_delta_3tier] RAE=0.2480 MAE=0.2256 R2=0.8390 r=0.9161 rho=0.8905

Tier usage: {'HIGH': 76, 'MED': 823, 'LOW': 1656, 'fallback': 1584}


In [11]:
# --- Blend sweep ---
best_alpha, best_rae = 1.0, m_dlt["RAE"]
print("\nBlend sweep (delta vs direct):")
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_delta + (1-alpha)*oof_direct
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae:
        best_rae, best_alpha = r, alpha

oof = best_alpha*oof_delta + (1-best_alpha)*oof_direct
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae:.4f}")

# Comparison vs nb104
nb104_path = DATA_PROCESSED / "oof_delta_similarity_tiers.npy"
if nb104_path.exists():
    oof104 = np.load(nb104_path)
    r104 = rae(y_tr, oof104)
    print(f"nb104 (3-tier, ECFP4 only) OOF RAE:   {r104:.4f}")
    print(f"nb111 (3-tier, enhanced feats) OOF RAE: {best_rae:.4f}")
    print(f"Delta: {best_rae - r104:+.4f}")


Blend sweep (delta vs direct):
  alpha=0.0  RAE=0.5598
  alpha=0.1  RAE=0.5278
  alpha=0.2  RAE=0.4958
  alpha=0.3  RAE=0.4639
  alpha=0.4  RAE=0.4320
  alpha=0.5  RAE=0.4002
  alpha=0.6  RAE=0.3685
  alpha=0.7  RAE=0.3370
  alpha=0.8  RAE=0.3059
  alpha=0.9  RAE=0.2757
  alpha=1.0  RAE=0.2480

Best blend alpha=1.0  OOF RAE=0.2480
nb104 (3-tier, ECFP4 only) OOF RAE:   0.2772
nb111 (3-tier, enhanced feats) OOF RAE: 0.2480
Delta: -0.0292


In [12]:
# --- Final test predictions ---
print("\nFitting final direct LGBM on all train...", flush=True)
m_final = lgb.train(LGBM_BASE, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

print("Running enhanced delta on test...", flush=True)
te_delta, te_tc, te_tu = predict_enhanced(
    fps4_te, fps4_tr,
    fps6_te, fps6_tr,
    maccs_te, maccs_tr,
    y_tr, phys_te, phys_tr,
    sim_te_tr, te_direct)
print(f"Test tier usage: {te_tc}")

te_preds = best_alpha*te_delta + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_enhanced_delta_3tier.npy", oof)
np.save(DATA_PROCESSED/"te_oof_enhanced_delta_3tier.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"111_delta_enhanced_features.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb111 OOF RAE = {best_rae:.4f} ***")


Fitting final direct LGBM on all train...


Running enhanced delta on test...


Test tier usage: {'HIGH': 91, 'MED': 369, 'LOW': 50, 'fallback': 3}
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\111_delta_enhanced_features.csv
Test: min=3.29 med=4.90 max=6.58

*** nb111 OOF RAE = 0.2480 ***
